# Rigid-Body Motions

Source orientation: printed pages 59-136, PDF pages 77-154. This notebook is an original visualization-first lesson inspired by the structure of *Modern Robotics: Mechanics, Planning, and Control* by Kevin M. Lynch and Frank C. Park. It does not quote or reproduce textbook prose, exercises, screenshots, or page crops.

## Chapter Question

How can a rigid displacement, a velocity, and a force all be represented without losing frame information?

This question is the thread for the chapter. The goal is not to memorize a list of formulas; it is to make the geometry inspectable. A robot mechanism is a physical machine, but the way we reason about it is through spaces, maps, constraints, tangents, and dual forces. The notebook keeps those objects visible. Every diagram and computation below is designed to answer a local question that a reader can check: what is moving, what is fixed, what map is being applied, what invariant should survive, and where can the model fail?

## Translation Guide

The source chapter or appendix is translated into computational language using these terms: SO(3), SE(3), homogeneous transform, twist, screw axis, adjoint, wrench. The important conversions for this notebook are:

- Rotation matrices encode orthonormal frames.
- A twist is a tangent vector to rigid motion.
- The adjoint is the bookkeeping map between frames.

The central habit is to name the representation and the invariant at the same time. Coordinates are useful only when we know what geometric object they represent. In this chapter, a 3 by 3 matrix is acceptable only if its columns remain an orthonormal right-handed frame, a 4 by 4 homogeneous transform is acceptable only if its rotation block stays in SO(3), and a 6-vector is meaningful only after we say whether it is a twist, a screw axis, or a wrench. The notebook therefore pairs each formula with a small experiment: a frame drawing, a screw path, a logarithm residual, a distance-preservation check, or a power-pairing check.

A useful computational translation is to treat SO(3) and SE(3) as spaces with their own arithmetic. Adding two rotation matrices is usually not a rotation; exponentiating a skew matrix is. A twist is not just a velocity list; it is the infinitesimal generator whose exponential gives a finite rigid displacement. The adjoint is the rule that moves that generator between frames while keeping the represented physical motion consistent. The dual coadjoint rule moves wrenches so that instantaneous power is unchanged.

## Route Through the Notebook

1. Establish the objects and conventions needed for rigid-body motions.
2. Build a concept dependency map so definitions have visible structure.
3. Generate the main visual lab: SO(3), SE(3), twists, screws, and wrenches.
4. Run a compact worked example that exposes an invariant.
5. Try an applied lab that changes a parameter and asks what should remain true.
6. Finish with sanity checks that assert artifact existence, image variation, and numerical margins.

This is a standalone course notebook. The PDF can be useful for historical context and exercises, but the lesson here is complete without it. Definitions are restated in fresh language, examples are computed from scratch, and visuals are generated by course-local code. When the notebook mentions a source span, it is a navigation reference rather than a dependency for comprehension.

## Visualization Storyboard

| Concept | Representation | Artifact | Inspection target |
| --- | --- | --- | --- |
| concept dependency map | directed graph | `artifacts/chapter-03-rigid-body-motions/figures/concept-dependency-map.png` | which definitions feed the chapter's computation |
| SO(3), SE(3), twists, screws, and wrenches | 3D frames and screw path | `artifacts/chapter-03-rigid-body-motions/figures/rigid-body-motion-lab.png` | how exponentials move frames while preserving rigid distances |
| numeric invariant check | residual bar chart and JSON summary | `artifacts/chapter-03-rigid-body-motions/figures/rigid-body-motion-checks.png` | small residuals, positive margins, or rank changes |

The first visual is a dependency map. It is intentionally simple: before computing anything, the reader should see which concepts support which later moves. The second visual is the main lab for this chapter. It turns the chapter's core geometry into something that can be inspected rather than imagined. The third visual is a numeric check: a path length and residual summary that can be tested after the figure is built.

## Working Principles

The most reliable way to learn robotics geometry is to move between three views. The first view is symbolic: equations name maps and constraints. The second view is numerical: a small instance exposes scale, rank, conditioning, and residuals. The third view is visual: geometry becomes a shape, path, ellipsoid, cone, graph, or surface. This notebook keeps all three views close together. If a symbolic statement is correct, the numerical experiment should produce the expected small residual or positive margin. If a visual is meaningful, it should make a specific invariant or failure mode easier to see.

For rigid-body motions, the relevant failure modes are not side details; they are part of the concept. Rotation coordinates have chart boundaries, logarithms become delicate near angle pi, and left-vs-right multiplication changes whether we are composing motions in space or body coordinates. A robust robotics model does not pretend those edges are absent. It names them, draws them, and then tests a small case so the reader can recognize the issue in later code.

## Applied Lab

Build a screw path and verify that the adjoint preserves twist/wrench power pairing.

In the lab, vary one parameter at a time and predict the direction of change before running the code. For example, increase the screw angle and predict that the path length should grow, set the pitch to zero and predict pure rotation about the axis, or move the axis point and predict that the same angular direction produces a different translation component. This prediction step is small, but it is what turns a figure into a mathematical instrument.

## Pitfalls To Watch

- Confusing active motion with coordinate change flips multiplication order.
- Ignoring frames makes the same vector appear to contradict itself.

These pitfalls are deliberately operational. If a computation becomes confusing, check frame labels, units, multiplication side, and the distinction between a physical object and its coordinates. Many robotics errors are not arithmetic mistakes; they are mismatches between a representation and the geometry it claims to encode.

## Takeaways

- Rigid motion lives on Lie groups, not flat vector spaces.
- Twists and wrenches are dual objects tied together by power.

By the end of this notebook, the reader should be able to explain the chapter's main object, build a small computed example, interpret the generated visual artifact, and state at least one numerical sanity check that would catch an implementation mistake.

In [ ]:
from pathlib import Path
import json
import math
import sys

import numpy as np

HERE = Path.cwd().resolve()
BOOK_ROOT = next(
    parent for parent in [HERE, *HERE.parents]
    if (parent / "AGENTS.md").exists() and (parent / "Mordern Robotics.pdf").exists()
)
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

from utils.artifacts import display_artifact, save_json
from utils.validation import assert_artifact, image_stats

print(f"BOOK_ROOT={BOOK_ROOT}")

In [ ]:
CHAPTER = {
  "number": 3,
  "slug": "chapter-03-rigid-body-motions",
  "title": "Rigid-Body Motions",
  "notebook": "03-rigid-body-motions.ipynb",
  "printed_start": 59,
  "printed_end": 136,
  "pdf_start": 77,
  "pdf_end": 154,
  "part_slug": "part-01-geometric-foundations",
  "part_title": "Geometric Foundations",
  "theme": "rigid",
  "visual_focus": "SO(3), SE(3), twists, screws, and wrenches",
  "visual_kind": "3D frames and screw path",
  "artifact_stem": "rigid-body-motion",
  "inspection_target": "how exponentials move frames while preserving rigid distances",
  "question": "How can a rigid displacement, a velocity, and a force all be represented without losing frame information?",
  "terms": [
    "SO(3)",
    "SE(3)",
    "homogeneous transform",
    "twist",
    "screw axis",
    "adjoint",
    "wrench"
  ],
  "translation": [
    "Rotation matrices encode orthonormal frames.",
    "A twist is a tangent vector to rigid motion.",
    "The adjoint is the bookkeeping map between frames."
  ],
  "lab": "Build a screw path and verify that the adjoint preserves twist/wrench power pairing.",
  "pitfalls": [
    "Confusing active motion with coordinate change flips multiplication order.",
    "Ignoring frames makes the same vector appear to contradict itself."
  ],
  "takeaways": [
    "Rigid motion lives on Lie groups, not flat vector spaces.",
    "Twists and wrenches are dual objects tied together by power."
  ]
}

from utils.visuals import build_storyboard
storyboard = build_storyboard(CHAPTER)
storyboard

In [ ]:
from utils.visuals import build_chapter_visuals

outputs = build_chapter_visuals(CHAPTER)
for artifact in outputs['figures']:
    display_artifact(artifact, width=760)
outputs['metrics']

## Worked Example

The worked example below is intentionally compact and reusable. It checks that a rotation exponential/logarithm round trip returns the same axis-angle vector, that an SE(3) screw exponential/logarithm round trip returns the same twist coordinates, that a rigid transform preserves distances between marked body points, and that the twist/wrench power pairing survives a frame transform. These are small tests, but they exercise the representation discipline needed before the later kinematics chapters build chains of transforms.

In [ ]:
from utils.lie import adjoint, coadjoint, screw_axis, se3_exp, se3_log, so3_exp, so3_log, transform_from, wrench_power

omega = np.array([0.2, -0.1, 0.35])
R = so3_exp(omega)
rotation_round_trip = np.linalg.norm(so3_log(R) - omega)
T = transform_from(R, np.array([0.25, -0.1, 0.4]))
body_points = np.array([[0.0, 0.0, 0.0], [0.6, -0.2, 0.35]])
moved_points = (T[:3, :3] @ body_points.T).T + T[:3, 3]
distance_error = abs(np.linalg.norm(body_points[1] - body_points[0]) - np.linalg.norm(moved_points[1] - moved_points[0]))
axis = screw_axis(np.array([0.2, -0.1, 0.0]), np.array([0.0, 0.0, 1.0]), pitch=0.18)
theta = 0.85
screw_round_trip = np.linalg.norm(se3_log(se3_exp(theta * axis)) - theta * axis)
twist = 0.35 * axis
wrench = np.array([0.3, 0.1, -0.2, 1.0, -0.4, 0.2])
power_before = wrench_power(twist, wrench)
power_after = wrench_power(adjoint(T) @ twist, coadjoint(T) @ wrench)
worked_example = {
    "rotation_log_round_trip": float(rotation_round_trip),
    "screw_log_round_trip": float(screw_round_trip),
    "distance_preservation_error": float(distance_error),
    "power_pairing_error": float(abs(power_before - power_after)),
    "adjoint_shape": list(adjoint(T).shape),
}
worked_example

## Applied Lab

The applied lab samples a finite screw motion over increasing angles. The numerical summary should agree with the visual lab: the rotation logarithm residual should stay small over the sampled range, the screw path should have nonzero length, and the final translation should reflect the selected pitch and axis offset.

In [ ]:
angles = np.linspace(0.0, 0.95 * np.pi, 60)
rotation_residuals = [np.linalg.norm(so3_log(so3_exp(np.array([0.0, 0.0, a]))) - np.array([0.0, 0.0, a])) for a in angles]
lab_axis = screw_axis(np.array([0.35, 0.0, 0.0]), np.array([0.0, 0.0, 1.0]), pitch=0.12)
lab_transforms = [se3_exp(a * lab_axis) for a in angles]
origins = np.array([item[:3, 3] for item in lab_transforms])
step_lengths = np.linalg.norm(np.diff(origins, axis=0), axis=1)
lab_summary = {
    "theme": CHAPTER["theme"],
    "max_rotation_residual": float(np.max(rotation_residuals)),
    "screw_path_length": float(np.sum(step_lengths)),
    "final_translation_norm": float(np.linalg.norm(origins[-1])),
}

lab_summary

## Sanity Checks

The final cell asserts that the generated artifacts exist, are large enough to be useful, and have nontrivial pixel variation. It also stores a JSON summary under the chapter's artifact subtree so the notebook leaves a machine-checkable trace.

In [ ]:
artifact_stats = {}
for artifact in outputs['figures']:
    resolved = assert_artifact(artifact)
    stats = image_stats(resolved)
    assert stats['pixel_std'] > 2.0, f'low variation image: {resolved}'
    artifact_stats[resolved.name] = stats
assert_artifact(outputs['checks'], min_size=100)
assert worked_example['rotation_log_round_trip'] < 1e-12
assert worked_example['screw_log_round_trip'] < 1e-12
assert worked_example['distance_preservation_error'] < 1e-12
assert worked_example['power_pairing_error'] < 1e-12
assert lab_summary['max_rotation_residual'] < 1e-12
assert lab_summary['screw_path_length'] > 0.1
sanity = {'chapter': CHAPTER['slug'], 'metrics': outputs['metrics'], 'worked_example': worked_example, 'lab_summary': lab_summary, 'artifact_stats': artifact_stats}
sanity_path = save_json(sanity, CHAPTER['slug'], 'checks', 'notebook-sanity.json')
display_artifact(sanity_path)
sanity